In [1]:
import jax.numpy as jnp
from sympy.vector import gradient


# TASK 1: The Forward Pass
def predict(w, b, X):
    # TODO: Calculate the dot product of X and w, then add b.
    return jnp.dot(X,w) + b
    # Hint: Use jnp.dot()


# TASK 2: The Loss Function
def mse_loss(w, b, X, y):
    # TODO: 1. Get the predictions using your predict() function.
    pred=predict(w,b,X)
    MSE=(y-pred)**2
    # TODO: 2. Calculate the squared errors: (predictions - y) ** 2
    # TODO: 3. Return the mean of these squared errors.
    return jnp.mean(MSE)
    # Hint: jnp.mean() is your friend.


In [6]:
# making the gradient
import jax
@jax.jit
def update_step(w, b, X, y, learning_rate):

    # TODO: 1. Create the gradient function using jax.grad (remember argnums=(0,1))
    calculate_graident=jax.grad(mse_loss,argnums=(0,1))

    # TODO: 2. Call your new gradient function to get grad_w and grad_b
    grad_w ,grad_b=calculate_graident(w,b,X,y)
    # TODO: 3. Update w: new_w = w - (learning_rate * grad_w)
    new_w= w - (learning_rate*grad_w)

    # TODO: 4. Update b: new_b = b - (learning_rate * grad_b)
    new_b= b - (learning_rate*grad_b)
    # TODO: 5. Return the new_w and new_b
    return new_w,new_b



In [7]:
import jax
import jax.numpy as jnp

# --- 1. The Forward Pass ---
def predict(w, b, X):
    return jnp.dot(X, w) + b

# --- 2. The Loss Function ---
def mse_loss(w, b, X, y):
    predictions = predict(w, b, X)
    squared_errors = (predictions - y) ** 2
    return jnp.mean(squared_errors)

# --- 3. The Update Engine (Compiled for Speed) ---
@jax.jit
def update_step(w, b, X, y, learning_rate):
    # Ask JAX to build the calculus derivatives for w (0th arg) and b (1st arg)
    calculate_gradient = jax.grad(mse_loss, argnums=(0, 1))

    # Run the math to get the exact correction amounts
    grad_w, grad_b = calculate_gradient(w, b, X, y)

    # Apply the corrections
    new_w = w - (learning_rate * grad_w)
    new_b = b - (learning_rate * grad_b)

    return new_w, new_b

# --- 4. The Training Loop ---
def train(X, y, epochs=1000, learning_rate=0.1):
    # Initialize our weights to zero
    n_features = X.shape[1]
    w = jnp.zeros(n_features)
    b = 0.0

    print("Starting training...")
    for epoch in range(epochs):
        # THE RELAY RACE: Pass the old state in, get the new state out!
        w, b = update_step(w, b, X, y, learning_rate)

        # Every 100 steps, let's peek at the error to ensure it's dropping
        if epoch % 100 == 0:
            current_loss = mse_loss(w, b, X, y)
            print(f"Epoch {epoch} | Loss: {current_loss:.4f}")

    return w, b

# --- 5. Run the Experiment! ---
if __name__ == "__main__":
    # Generate fake data: 100 samples, 2 features
    key = jax.random.PRNGKey(42)
    X = jax.random.normal(key, (100, 2))

    # Let's create a secret rule: y = 3.0*x1 + 5.0*x2 - 2.0
    # We will add a tiny bit of noise so it's realistic
    true_w = jnp.array([3.0, 5.0])
    true_b = -2.0
    y = jnp.dot(X, true_w) + true_b + 0.1 * jax.random.normal(key, (100,))

    # Let the model figure out the secret rule!
    final_w, final_b = train(X, y, epochs=500, learning_rate=0.1)

    print("\n--- Final Results ---")
    print(f"True Weights: {true_w} | Learned Weights: {final_w}")
    print(f"True Bias:  {true_b} | Learned Bias:  {final_b:.4f}")

Starting training...
Epoch 0 | Loss: 20.0173
Epoch 100 | Loss: 0.0078
Epoch 200 | Loss: 0.0078
Epoch 300 | Loss: 0.0078
Epoch 400 | Loss: 0.0078

--- Final Results ---
True Weights: [3. 5.] | Learned Weights: [2.9835758 4.9867334]
True Bias:  -2.0 | Learned Bias:  -1.9970


In [8]:
import time

if __name__ == "__main__":
    # Generate fake data
    key = jax.random.PRNGKey(42)
    X = jax.random.normal(key, (100, 2))

    true_w = jnp.array([3.0, 5.0])
    true_b = -2.0
    y = jnp.dot(X, true_w) + true_b + 0.1 * jax.random.normal(key, (100,))

    # --- THE TIMING EXPERIMENT ---

    print("--- RUN 1: Building the Factory (JIT Compilation + Training) ---")
    start_time = time.time()
    # The first time this runs, JAX has to translate update_step into C++/XLA
    final_w, final_b = train(X, y, epochs=500, learning_rate=0.1)
    run_1_time = (time.time() - start_time) * 1000
    print(f"Run 1 took: {run_1_time:.2f} milliseconds\n")

    print("--- RUN 2: Using the Built Factory (Pure Training) ---")
    start_time = time.time()
    # The second time, the C++ blueprint already exists in memory.
    # It just dumps the data in and runs the math instantly.
    final_w, final_b = train(X, y, epochs=500, learning_rate=0.1)
    run_2_time = (time.time() - start_time) * 1000
    print(f"Run 2 took: {run_2_time:.2f} milliseconds")

    print("\n--------------------------------------------------")
    print(f"JAX is actually {run_1_time / run_2_time:.1f}x faster once compiled!")

--- RUN 1: Building the Factory (JIT Compilation + Training) ---
Starting training...
Epoch 0 | Loss: 20.0173
Epoch 100 | Loss: 0.0078
Epoch 200 | Loss: 0.0078
Epoch 300 | Loss: 0.0078
Epoch 400 | Loss: 0.0078
Run 1 took: 8.00 milliseconds

--- RUN 2: Using the Built Factory (Pure Training) ---
Starting training...
Epoch 0 | Loss: 20.0173
Epoch 100 | Loss: 0.0078
Epoch 200 | Loss: 0.0078
Epoch 300 | Loss: 0.0078
Epoch 400 | Loss: 0.0078
Run 2 took: 6.99 milliseconds

--------------------------------------------------
JAX is actually 1.1x faster once compiled!
